In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd

Matplotlib is building the font cache; this may take a moment.


# IMP

## Look at IMP dataset

In [2]:
imp_path = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/")
items = list(imp_path.glob("*"))
items

[PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/test'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/train'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/val'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/.DS_Store'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/test.txt'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/train.txt'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/train_0_001.txt'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/train_0_005.txt'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/train_0_005_ot.txt'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/

In [25]:
train_dir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/train')
val_dir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/val')
test_dir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/test')

In [29]:
train_files = [elem for elem in train_dir.glob("*") if ".DS_Store" not in str(elem)]
val_files = [elem for elem in val_dir.glob("*") if ".DS_Store" not in str(elem)]
test_files = [elem for elem in test_dir.glob("*") if ".DS_Store" not in str(elem)]

train_tifs = [f for f in train_files if str(f).endswith(".tif")]
val_tifs = [f for f in val_files if str(f).endswith(".tif")]
test_tifs = [f for f in test_files if str(f).endswith(".tif")]

print(f"Total train files: {len(train_files)}, num tifs: {len(train_tifs)}")
print(f"Total val files: {len(val_files)}, num tifs: {len(val_tifs)}")
print(f"Total test files: {len(test_files)}, num tifs: {len(test_tifs)}")

Total train files: 21126, num tifs: 21126
Total val files: 4044, num tifs: 4041
Total test files: 2832, num tifs: 2832


In [22]:
for split_files in [train_tifs, val_tifs, test_tifs]:
    negative_files = [f for f in split_files if "negative" in str(f)]
    target_files = [f for f in split_files if "target" in str(f)]
    print(f"Found {len(negative_files)} negatives, {len(target_files)} targets")
    for file in negative_files[:100]:
        train_ds = xr.open_dataset(file)
        if int(train_ds.sizes['band']) != 1: print(">1 band found in negative files")
    for file in target_files[:100]:
        train_ds = xr.open_dataset(file)
        if int(train_ds.sizes['band']) != 1: print(">1 band found in target files")

Found 14094 negatives, 7032 targets
Found 2571 negatives, 1470 targets
Found 2001 negatives, 831 targets


In [21]:
print(len(train_tifs))
print(len(val_tifs))
print(len(test_tifs))

21126
4041
2832


## Agent code

### Snippet 1

In [30]:
from pathlib import Path
from collections import Counter, defaultdict
import re

root = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset")

for split in ["train", "val", "test"]:
  split_dir = root / split
  files = [p for p in split_dir.iterdir() if p.name != ".DS_Store"]

  exts = Counter(p.suffix.lower() for p in files)
  labels = Counter(
      "target" if "__target_" in p.name else
      "negative" if "__negative_" in p.name else
      "unknown"
      for p in files
  )

  print(f"\n{split}")
  print("num files:", len(files))
  print("extensions:", exts)
  print("label/name patterns:", labels)
  print("examples:")
  for p in files[:5]:
      print(" ", p.name)


train
num files: 21126
extensions: Counter({'.tif': 21126})
label/name patterns: Counter({'negative': 14094, 'target': 7032})
examples:
  M102414599RE.ech.cog__negative_00006__c1394_r31356_img.tif
  M102414599RE.ech.cog__negative_00006__c1394_r31356_mask.tif
  M102414599RE.ech.cog__negative_00006__c1394_r31356_mask_orig.tif
  M104354874RE.ech.cog__negative_00003__c923_r26236_img.tif
  M104354874RE.ech.cog__negative_00003__c923_r26236_mask.tif

val
num files: 4044
extensions: Counter({'.tif': 4041, '.xml': 3})
label/name patterns: Counter({'negative': 2571, 'target': 1473})
examples:
  M102414599RE.ech.cog__negative_00003__c1991_r71814_img.tif
  M102414599RE.ech.cog__negative_00003__c1991_r71814_mask.tif
  M102414599RE.ech.cog__negative_00003__c1991_r71814_mask_orig.tif
  M108951277RE.ech.cog__negative_00047__c1589_r8567_img.tif
  M108951277RE.ech.cog__negative_00047__c1589_r8567_mask.tif

test
num files: 2832
extensions: Counter({'.tif': 2832})
label/name patterns: Counter({'negative'

### Snippet 2

In [31]:
import rasterio
import numpy as np

sample = next((root / "train").glob("*.tif"))

with rasterio.open(sample) as src:
  print("path:", sample)
  print("count:", src.count)
  print("shape:", src.height, src.width)
  print("dtype:", src.dtypes)
  print("nodata:", src.nodata)
  print("tags:", src.tags())
  arr = src.read()
  print("array shape:", arr.shape)
  print("min/max/mean/std:", np.nanmin(arr), np.nanmax(arr), np.nanmean(arr), np.nanstd(arr))

path: /explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset/train/M102414599RE.ech.cog__negative_00006__c1394_r31356_img.tif
count: 1
shape: 256 256
dtype: ('float32',)
nodata: -3.4028226550889045e+38
tags: {'AREA_OR_POINT': 'Area'}
array shape: (1, 256, 256)
min/max/mean/std: -0.0005304716 0.012841965 0.0009114246 0.0011277042


### Snippet 3

In [32]:
from pathlib import Path
import rasterio
import numpy as np
from collections import Counter

root = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset")

for split in ["train", "val", "test"]:
  print(f"\n{split}")
  for pattern in ["*negative*_mask.tif", "*target*_mask.tif", "*target*_mask_orig.tif"]:
      vals = Counter()
      files = sorted((root / split).glob(pattern))[:20]
      for p in files:
          with rasterio.open(p) as src:
              arr = src.read(1)
          vals.update(np.unique(arr).tolist())
      print(pattern, vals)


train


/tmp/ipykernel_556907/477597537.py:15: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1)


*negative*_mask.tif Counter({0: 20})
*target*_mask.tif Counter({0: 20, 1: 20})
*target*_mask_orig.tif Counter({0: 20, 1: 20})

val
*negative*_mask.tif Counter({0: 20})
*target*_mask.tif Counter({0: 20, 1: 20})
*target*_mask_orig.tif Counter({0: 20, 1: 20})

test
*negative*_mask.tif Counter({0: 20})
*target*_mask.tif Counter({0: 20, 1: 20})
*target*_mask_orig.tif Counter({0: 20, 1: 20})


### Snippet 4

In [33]:
for split in ["train", "val", "test"]:
  split_dir = root / split
  img_stems = {p.name.removesuffix("_img.tif") for p in split_dir.glob("*_img.tif")}
  mask_stems = {p.name.removesuffix("_mask.tif") for p in split_dir.glob("*_mask.tif")}
  orig_stems = {p.name.removesuffix("_mask_orig.tif") for p in split_dir.glob("*_mask_orig.tif")}

  print(f"\n{split}")
  print("img:", len(img_stems), "mask:", len(mask_stems), "mask_orig:", len(orig_stems))
  print("missing masks:", len(img_stems - mask_stems))
  print("missing imgs:", len(mask_stems - img_stems))
  print("missing mask_orig:", len(img_stems - orig_stems))


train
img: 7042 mask: 7042 mask_orig: 7042
missing masks: 0
missing imgs: 0
missing mask_orig: 0

val
img: 1347 mask: 1347 mask_orig: 1347
missing masks: 0
missing imgs: 0
missing mask_orig: 0

test
img: 944 mask: 944 mask_orig: 944
missing masks: 0
missing imgs: 0
missing mask_orig: 0


# NAC

## Look at NAC dataset

In [6]:
LFM_DIR = Path("/explore/nobackup/projects/lfm/")

In [7]:
nac_path = LFM_DIR / "processed_data/Lunar/data_release/NAC_craters_coco_release_final/"
contents = list(nac_path.glob("*"))
contents

[PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/DTM'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/PHO'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/splits'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/metadata.parquet'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/nac_handlabeled_annotations.json')]

In [3]:
pq_path = next(nac_path.glob("*.parquet"))
# pq_path = [p for p in [str(elem) for elem in contents] if ".parquet" in p][0]
df = pd.read_parquet(pq_path)
df.head()

,PRODUCT_ID,PHO_TILE,DTM_TILE,LTM_CODE,EMISSION_ANGLE,INCIDENCE_ANGLE,PHASE_ANGLE,SUB_SOLAR_GROUND_AZIMUTH,SUB_SOLAR_LATITUDE,SUB_SOLAR_LONGITUDE,UPPER_LEFT_LONGITUDE,LOWER_RIGHT_LONGITUDE,UPPER_LEFT_LATITUDE,LOWER_RIGHT_LATITUDE,CENTER_LONGITUDE,CENTER_LATITUDE,DATASET
0,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.831589,-11.823145,-0.457667,-0.449225,-11.827367,-0.453446,train
1,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.823145,-11.814701,-0.457667,-0.449225,-11.818923,-0.453446,train
2,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.814701,-11.806258,-0.457667,-0.449225,-11.810480,-0.453446,train
3,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.806258,-11.797814,-0.457667,-0.449225,-11.802036,-0.453446,train
4,M1308402325RE,PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R...,LTM_22S,1.15,74.62,73.47,268.818685,-1.51,-86.45,-11.831589,-11.823145,-0.466109,-0.457667,-11.827367,-0.461888,train


In [8]:
first_item = dict(df.iloc[0])
first_pho_tile = nac_path / first_item['PHO_TILE']
first_dtm_tile =  nac_path / first_item['DTM_TILE']
first_pho_tile, first_dtm_tile

(PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/PHO/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_PHO_r0_c0.nc'),
 PosixPath('/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final/DTM/A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_DTM_r0_c0.nc'))

In [11]:
pho_ds = xr.open_dataset(first_pho_tile)
print(pho_ds)

<xarray.Dataset> Size: 266kB
Dimensions:      (y: 256, x: 256)
Coordinates:
  * y            (y) float64 2kB -1.362e+04 -1.362e+04 ... -1.388e+04 -1.388e+04
  * x            (x) float64 2kB 5.099e+06 5.099e+06 ... 5.099e+06 5.099e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (y, x) float32 262kB ...
Attributes:
    crs:      PROJCS["Equirectangular_Moon",GEOGCS["GCS_Moon",DATUM["D_Moon",...


In [12]:
dtm_ds = xr.open_dataset(first_dtm_tile)
print(dtm_ds)

<xarray.Dataset> Size: 266kB
Dimensions:      (y: 256, x: 256)
Coordinates:
  * y            (y) float64 2kB -1.362e+04 -1.362e+04 ... -1.388e+04 -1.388e+04
  * x            (x) float64 2kB 5.099e+06 5.099e+06 ... 5.099e+06 5.099e+06
    spatial_ref  int64 8B ...
Data variables:
    band_data    (y, x) float32 262kB ...
Attributes:
    crs:      PROJCS["Equirectangular_Moon",GEOGCS["GCS_Moon",DATUM["D_Moon",...


## Longer check for Codex

### First check

In [13]:
from pathlib import Path
import json
import pandas as pd

nac_path = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final")

print("Top level:")
for p in sorted(nac_path.iterdir()):
  print(p.name, "dir" if p.is_dir() else "file")

df = pd.read_parquet(nac_path / "metadata.parquet")
print("\nColumns:", list(df.columns))
print("\nSplit counts:")
print(df["DATASET"].value_counts(dropna=False))

print("\nPHO count:", len(list((nac_path / "PHO").glob("*.nc"))))
print("DTM count:", len(list((nac_path / "DTM").glob("*.nc"))))

print("\nSplits dir:")
for p in sorted((nac_path / "splits").glob("*")):
  print(p.name)

with open(nac_path / "nac_handlabeled_annotations.json") as f:
  coco = json.load(f)

print("\nCOCO keys:", coco.keys())
print("images:", len(coco.get("images", [])))
print("annotations:", len(coco.get("annotations", [])))
print("categories:", coco.get("categories", [])[:5])
print("\nFirst image:", coco.get("images", [None])[0])
print("\nFirst annotation:", coco.get("annotations", [None])[0])

Top level:
DTM dir
PHO dir
metadata.parquet file
nac_handlabeled_annotations.json file
splits dir

Columns: ['PRODUCT_ID', 'PHO_TILE', 'DTM_TILE', 'LTM_CODE', 'EMISSION_ANGLE', 'INCIDENCE_ANGLE', 'PHASE_ANGLE', 'SUB_SOLAR_GROUND_AZIMUTH', 'SUB_SOLAR_LATITUDE', 'SUB_SOLAR_LONGITUDE', 'UPPER_LEFT_LONGITUDE', 'LOWER_RIGHT_LONGITUDE', 'UPPER_LEFT_LATITUDE', 'LOWER_RIGHT_LATITUDE', 'CENTER_LONGITUDE', 'CENTER_LATITUDE', 'DATASET']

Split counts:
DATASET
train    645
test      62
val       59
Name: count, dtype: int64

PHO count: 766
DTM count: 766

Splits dir:
test.json
train.json
val.json

COCO keys: dict_keys(['info', 'images', 'categories', 'annotations'])
images: 766
annotations: 97104
categories: [{'id': 1, 'name': 'crater', 'supercategory': None}]

First image: {'id': 205, 'file_name': 'A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_PHO_r0_c0.nc', 'height': 256, 'width': 256, 'license': 0}

First annotation: {'id': 10329, 'image_id': 205, 'category_id': 1, 'segmentation': [[1, 237, 2, 238

### Second check

In [14]:
import json
from pathlib import Path

nac_path = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/NAC_craters_coco_release_final")

for split in ["train", "val", "test"]:
  with open(nac_path / "splits" / f"{split}.json") as f:
      obj = json.load(f)
  print(split, type(obj))
  if isinstance(obj, dict):
      print(obj.keys())
      for k, v in obj.items():
          if isinstance(v, list):
              print(k, len(v), v[0] if v else None)
  elif isinstance(obj, list):
      print(len(obj), obj[0] if obj else None)
  print()

train <class 'dict'>
dict_keys(['info', 'images', 'categories', 'annotations'])
images 645 {'id': 205, 'file_name': 'A15SIVB_PHO_E009S3481_LC_box1_M1308402325R_PHO_r0_c0.nc', 'height': 256, 'width': 256, 'license': 0}
categories 1 {'id': 1, 'name': 'crater', 'supercategory': None}
annotations 84536 {'id': 10329, 'image_id': 205, 'category_id': 1, 'segmentation': [[1, 237, 2, 238, 4, 239, 6, 241, 7, 242, 9, 243, 11, 244, 13, 244, 15, 245, 17, 246, 19, 246, 21, 247, 23, 247, 25, 248, 27, 248, 29, 248, 31, 248, 33, 248, 35, 248, 37, 248, 39, 248, 41, 247, 43, 247, 45, 246, 47, 246, 49, 245, 51, 244, 53, 244, 54, 243, 56, 242, 58, 241, 60, 239, 61, 238, 63, 237, 64, 236, 66, 234, 67, 233, 69, 231, 70, 230, 71, 228, 72, 226, 74, 225, 75, 223, 76, 221, 76, 219, 77, 217, 78, 215, 79, 213, 79, 211, 80, 209, 80, 207, 80, 205, 80, 203, 81, 201, 81, 199, 81, 197, 80, 195, 80, 193, 80, 191, 80, 189, 79, 187, 79, 185, 78, 183, 77, 181, 76, 179, 76, 178, 75, 176, 74, 174, 72, 172, 71, 171, 70, 169, 

# WAC

## iseg label comparison

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scratch.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

from lfm.full_model.all_tasks.utils import create_timestamped_output_dir, plot_instance_label_comparison

KAGUYA_ISEG_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg")
NEW_ISEG_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")
ISEG_LABEL_COMPARISON_OUTPUT_DIR = create_timestamped_output_dir(NOTEBOOK_DIR / "outputs" / "iseg_label_comparison")

plot_instance_label_comparison(
    kaguya_root=KAGUYA_ISEG_ROOT,
    split_data_root=NEW_ISEG_ROOT,
    output_dir=ISEG_LABEL_COMPARISON_OUTPUT_DIR,
    n_samples=8,
    filename="iseg_label_comparison.png",
    display_method="minmax",
    dpi=200,
)

## iseg sanity test

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scratch.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]
GRAHA_ROOT = LFM_ROOT / "lfm" / "full_model" / "graha-lunar-fm"

PRETRAIN_DIR = Path(
    "/explore/nobackup/projects/lfm/gabby/Lunar-FM/experiments/"
    "lunarfm_base_dual_full_nas_no_nans_256_256_lr1e-4_wd0.05"
).resolve()
BACKBONE_WEIGHTS = PRETRAIN_DIR / "checkpoints/checkpoint_weights_final.pt"
BACKBONE_CFG = PRETRAIN_DIR / "full_config.yaml"
MODALITY_INFO = PRETRAIN_DIR / "modality_info.yaml"
NORMALIZED_WAC_DATA_RANGE = [-1.0, 1.0]

for import_path in [GRAHA_ROOT, LFM_ROOT]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from lfm.full_model.inst_seg.instance_mask_datamodule import (
    LunarInstanceMaskSegmentationDatamodule,
    LunarObjectDetectionInstanceMaskDatamodule,
)
from lfm.full_model.all_tasks.utils import create_timestamped_output_dir, plot_instance_batch_sanity
from lfm.full_model.all_tasks.utils.utils import ensure_data_symlink

print("Notebook directory:", NOTEBOOK_DIR)
print("LFM root:", LFM_ROOT)
print("Graha/Lunar-FM code root:", GRAHA_ROOT)
print("Backbone weights:", BACKBONE_WEIGHTS)

In [ ]:
ISEG_DATA_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")  # expects train/val/test/{chips,labels}
MASK_SHIFT = (0, 0)  # (x_pixels, y_pixels): positive moves labels right/down
ISEG_OUTPUT_DIR = create_timestamped_output_dir(NOTEBOOK_DIR / "outputs" / "instance_sanity")

iseg_datamodule = LunarInstanceMaskSegmentationDatamodule(
    data_root=ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=5,
    num_workers=0,
    mask_shift=MASK_SHIFT,
)

plot_instance_batch_sanity(
    iseg_datamodule,
    output_dir=ISEG_OUTPUT_DIR,
    split="train",
    n_samples=5,
)

### ObjectDetectionTask target sanity test

This checks the true instance target format expected by TerraTorch `ObjectDetectionTask`: `image`, `boxes`, `labels`, and `masks`.

In [ ]:
OD_ISEG_DATA_ROOT = ISEG_DATA_ROOT

od_iseg_datamodule = LunarObjectDetectionInstanceMaskDatamodule(
    data_root=OD_ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=4,
    num_workers=0,
    mask_shift=MASK_SHIFT,
    target_box_format="xyxy",  # pixel xyxy for mask-rcnn
)
od_iseg_datamodule.setup("fit")
od_batch = next(iter(od_iseg_datamodule.train_dataloader()))

print("batch keys:", od_batch.keys())
print("image:", tuple(od_batch["image"].shape), od_batch["image"].dtype)
print("boxes per image:", [tuple(x.shape) for x in od_batch["boxes"]])
print("labels per image:", [tuple(x.shape) for x in od_batch["labels"]])
print("masks per image:", [tuple(x.shape) for x in od_batch["masks"]])
print("first filename:", od_batch["filename"][0])

for i, (boxes, labels, masks) in enumerate(zip(od_batch["boxes"], od_batch["labels"], od_batch["masks"])):
    assert boxes.ndim == 2 and boxes.shape[-1] == 4
    assert labels.ndim == 1
    assert masks.ndim == 3 and masks.shape[-2:] == od_batch["image"].shape[-2:]
    assert boxes.shape[0] == labels.shape[0] == masks.shape[0]
    if boxes.numel():
        assert bool((boxes[:, 2] > boxes[:, 0]).all())
        assert bool((boxes[:, 3] > boxes[:, 1]).all())
print("ObjectDetectionTask target sanity check passed.")